In [11]:
import json
import os
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import ftfy
import pandas as pd
import requests
from lingua import Language, LanguageDetectorBuilder
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://openlibrary.org"
HEADERS = {"User-Agent": "AdvancedDataMining/1.0 (your-email@example.com)"}
# `language:eng` makes Open Library return the matching English edition of each work
SEARCH_QUERY = "subject:fiction language:eng"
SEARCH_FIELDS = "key,title,author_name,description,subject,editions,editions.key,editions.title,editions.language"
N_BOOKS = 20_000  # target number of English books (after filtering)
BATCH_SIZE = 50
REQUEST_DELAY = 1.0

RAW_DIR = Path("data/raw/batches_eng")
PROGRESS_PATH = RAW_DIR / "_progress.json"
PROCESSED_DIR = Path("data/processed")
JSON_PATH = PROCESSED_DIR / "books.json"
PARQUET_PATH = PROCESSED_DIR / "books.parquet"

session = requests.Session()
session.headers.update(HEADERS)
retry = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retry))

In [6]:
# English-only filter:
# 1. Open Library must return an edition catalogued as English only (language == ["eng"]);
#    its title is used as the book name, e.g. "O Alquimista" -> "The Alchemist".
# 2. The (work-level) description must be detected as English.
ENGLISH_CODE = "eng"

# Languages that steal short/mixed English texts, excluded from description detection
NOISY_LANGUAGES = [
    Language.LATIN, Language.ESPERANTO, Language.SOMALI, Language.SWAHILI, Language.TSWANA,
    Language.SOTHO, Language.XHOSA, Language.ZULU, Language.TSONGA, Language.YORUBA,
    Language.SHONA, Language.GANDA, Language.MAORI, Language.TAGALOG, Language.WELSH,
    Language.IRISH, Language.AFRIKAANS, Language.BASQUE, Language.ALBANIAN, Language.NYNORSK,
    Language.CATALAN,
]
description_detector = LanguageDetectorBuilder.from_all_languages_without(*NOISY_LANGUAGES).build()


def get_description(doc):
    description = doc.get("description")
    if isinstance(description, dict):
        description = description.get("value")
    return (description or "").strip()


def get_english_edition(doc):
    """The edition Open Library matched for `language:eng`, if it is English-only."""
    editions = (doc.get("editions") or {}).get("docs", [])
    if editions and editions[0].get("language") == [ENGLISH_CODE] and editions[0].get("title"):
        return editions[0]
    return None


def is_latin_script(text):
    return all(not ch.isalpha() or "LATIN" in unicodedata.name(ch, "") for ch in text)


def is_english_description(description):
    return bool(description) and description_detector.detect_language_of(description[:2000]) == Language.ENGLISH


def is_english_book(doc):
    edition = get_english_edition(doc)
    return (
        edition is not None
        and is_latin_script(edition["title"])
        and is_english_description(get_description(doc))
    )


# Sanity check
english_edition = {"docs": [{"title": "Dune", "language": ["eng"]}]}
bilingual_edition = {"docs": [{"title": "El Principito / The Little Prince", "language": ["spa", "eng"]}]}
for label, doc in [
    ("English", {"editions": english_edition, "description": "Set on the desert planet Arrakis, Dune is the story of Paul Atreides."}),
    ("bilingual edition", {"editions": bilingual_edition, "description": "A pilot stranded in the desert meets a young prince."}),
    ("French description", {"editions": english_edition, "description": "Le Petit Prince est une œuvre de langue française."}),
]:
    print(f"{label:20} -> {is_english_book(doc)}")

English              -> True
bilingual edition    -> False
French description   -> False


In [7]:
def batch_path(batch_no):
    return RAW_DIR / f"batch_{batch_no:05d}.jsonl"


def atomic_write(path, text):
    """Write via a temp file + rename, so a crash never leaves a half-written file."""
    tmp_path = path.with_name(path.name + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    tmp_path.replace(path)


def batch_docs(path):
    docs = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            docs.extend(json.loads(json.loads(line)["response"]).get("docs", []))
    return docs


def load_progress():
    """batch_no -> list of English work keys found in that batch."""
    if PROGRESS_PATH.exists():
        return {int(k): v for k, v in json.loads(PROGRESS_PATH.read_text(encoding="utf-8")).items()}
    return {}


def save_progress(progress):
    atomic_write(PROGRESS_PATH, json.dumps({str(k): v for k, v in sorted(progress.items())}))


def fetch_batch(batch_no):
    """Download one search page and land the raw, unedited response as JSONL."""
    offset = batch_no * BATCH_SIZE
    response = session.get(
        f"{BASE_URL}/search.json",
        params={
            "q": SEARCH_QUERY,
            "fields": SEARCH_FIELDS,
            "sort": "readinglog",
            "limit": BATCH_SIZE,
            "offset": offset,
        },
        timeout=(10, 120),
    )
    response.raise_for_status()
    record = {
        "batch_no": batch_no,
        "offset": offset,
        "url": response.url,
        "status_code": response.status_code,
        "fetched_at": datetime.now(timezone.utc).isoformat(),
        "response": response.text,
    }
    atomic_write(batch_path(batch_no), json.dumps(record, ensure_ascii=False) + "\n")
    return json.loads(response.text).get("docs", [])


def download_all():
    """Fetch batches until N_BOOKS unique English books are landed.

    Safe to stop at any time: finished batches are kept on disk and skipped on the next run.
    """
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for tmp in RAW_DIR.glob("*.tmp"):
        tmp.unlink()

    progress = load_progress()
    english_keys = set()
    batch_no = 0
    started = time.time()
    while len(english_keys) < N_BOOKS:
        path = batch_path(batch_no)
        if batch_no in progress and path.exists():
            english_keys.update(progress[batch_no])
            batch_no += 1
            continue

        if path.exists():
            docs = batch_docs(path)  # landed, but progress not saved before the stop
        else:
            docs = fetch_batch(batch_no)
            time.sleep(REQUEST_DELAY)
        if not docs:
            print("No more search results.")
            break

        progress[batch_no] = [doc["key"] for doc in docs if is_english_book(doc)]
        save_progress(progress)
        english_keys.update(progress[batch_no])
        print(
            f"batch {batch_no:05d} | +{len(progress[batch_no]):>2}/{len(docs)} English"
            f" | {len(english_keys):>6}/{N_BOOKS} | {time.time() - started:6.0f}s",
            flush=True,
        )
        batch_no += 1
    print(f"Done: {len(english_keys)} English books from {batch_no} batches in {RAW_DIR}")

In [8]:
download_all()

batch 00000 | +47/50 English |     47/20000 |      4s
batch 00001 | +48/50 English |     95/20000 |      7s
batch 00002 | +46/50 English |    141/20000 |     11s
batch 00003 | +49/50 English |    190/20000 |     14s
batch 00004 | +47/50 English |    237/20000 |     17s
batch 00005 | +46/50 English |    283/20000 |     20s
batch 00006 | +47/50 English |    330/20000 |     23s
batch 00007 | +46/50 English |    376/20000 |     26s
batch 00008 | +45/50 English |    421/20000 |     29s
batch 00009 | +47/50 English |    468/20000 |     32s
batch 00010 | +47/50 English |    515/20000 |     35s
batch 00011 | +46/50 English |    561/20000 |     38s
batch 00012 | +46/50 English |    607/20000 |     41s
batch 00013 | +46/50 English |    653/20000 |     44s
batch 00014 | +48/50 English |    701/20000 |     47s
batch 00015 | +48/50 English |    749/20000 |     50s
batch 00016 | +47/50 English |    796/20000 |     53s
batch 00017 | +48/50 English |    844/20000 |     56s
batch 00018 | +48/50 English

In [12]:
# Invisible/garbage characters: leftover C1 controls (after ftfy), zero-width, BOM, soft hyphen, replacement char
INVISIBLE_CHARS = re.compile("[\x80-\x9f\u200b-\u200f\u2060\ufeff\u00ad\ufffd]")
# Form feed, vertical tab, non-breaking and other exotic spaces -> regular space
UNICODE_SPACES = re.compile("[\x0b\x0c\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]")
# Some catalogue records store umlauts as vowel + byte 0x98 ("Ma\x98dchen" -> "Mädchen")
STRAY_UMLAUT = re.compile("([AaEeIiOoUu])\x98")


def clean_text(text):
    """Return clean NFC-normalised Unicode: fixes mojibake, HTML entities, control chars and line breaks."""
    if not isinstance(text, str):
        return text
    text = STRAY_UMLAUT.sub("\\1\u0308", text)
    text = ftfy.fix_text(text, uncurl_quotes=False, normalization="NFC")
    text = INVISIBLE_CHARS.sub("", text)
    text = UNICODE_SPACES.sub(" ", text)
    return unicodedata.normalize("NFC", text).strip()


def clean_list(values):
    """Clean every string, dropping empties and duplicates created by cleaning (order preserved)."""
    return list(dict.fromkeys(v for v in map(clean_text, values or []) if v))


def parse_raw():
    """Re-parse all landed raw batches into the working DataFrame (English books only, clean Unicode)."""
    progress = load_progress()
    rows = []
    for path in sorted(RAW_DIR.glob("batch_*.jsonl")):
        batch_no = int(path.stem.split("_")[1])
        docs = batch_docs(path)
        english = set(progress[batch_no]) if batch_no in progress else {d["key"] for d in docs if is_english_book(d)}
        for doc in docs:
            if doc["key"] not in english:
                continue
            edition = get_english_edition(doc)
            rows.append(
                {
                    "work_key": doc["key"],
                    "edition_key": edition.get("key"),
                    "author": ", ".join(clean_list(doc.get("author_name"))) or None,
                    "name": clean_text(edition["title"]),
                    "description": clean_text(get_description(doc)),
                    "genres": clean_list(doc.get("subject")),
                }
            )
    df = pd.DataFrame(rows, columns=["work_key", "edition_key", "author", "name", "description", "genres"])
    return df.drop_duplicates("work_key", keep="first").head(N_BOOKS).reset_index(drop=True)


df = parse_raw()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# force_ascii=False writes real UTF-8 characters instead of \uXXXX escapes
df.to_json(JSON_PATH, orient="records", force_ascii=False, indent=2)
df.to_parquet(PARQUET_PATH, engine="pyarrow", index=False)
print(f"Saved {len(df)} books -> {JSON_PATH} and {PARQUET_PATH}")
df.head()

Saved 20000 books -> data/processed/books.json and data/processed/books.parquet


,work_key,edition_key,author,name,description,genres
0,/works/OL18020194W,/books/OL27213498M,Colleen Hoover,It Ends With Us,"Lily hasn’t always had it easy, but that’s nev...","[Fiction, Romance, Contemporary, Life change e..."
1,/works/OL82563W,/books/OL59004869M,J. K. Rowling,Harry Potter and the Philosopher's Stone,"Turning the envelope over, his hand trembling,...","[series:Harry_Potter, Ghosts, Monsters, Vampir..."
2,/works/OL27733867W,/books/OL49140417M,Colleen Hoover,It Starts with Us,"**Before It Ends with Us, it started with Atla...","[Contemporary, Romance, Friendship, Fiction, Y..."
3,/works/OL24390422W,/books/OL37985553M,Ana Huang,Twisted Love,"He has a heart of ice … but for her, he'd burn...","[Fiction, romance, multicultural & interracial..."
4,/works/OL257943W,/books/OL58636603M,George R. R. Martin,A Game of Thrones,***A Game of Thrones*** is the inaugural novel...,"[Adult, Action & Adventure, Fantasy, High Fant..."
